# 5. Interactive Launch-Site Map with Folium

In [1]:
from pathlib import Path

print(list(Path("/content").rglob("spacex_launches_cleaned.csv")))

import pandas as pd

csv_path = "/content/spacex_launches_cleaned.csv"
df = pd.read_csv(csv_path)

print("Dataset loaded:", df.shape)
print("Available columns:", df.columns.tolist())
display(df.head())

from google.colab import files
import pandas as pd
import io

uploaded = files.upload()

csv_files = [
    name for name in uploaded
    if name.lower().endswith(".csv")
]

if not csv_files:
    raise FileNotFoundError(
        "No CSV uploaded. Please select spacex_launches_cleaned.csv."
    )

# Prefer the cleaned CSV; otherwise use the available CSV
csv_name = next(
    (name for name in csv_files
     if name.lower() == "spacex_launches_cleaned.csv"),
    csv_files[0]
)

df = pd.read_csv(io.BytesIO(uploaded[csv_name]))

print("Dataset loaded:", df.shape)
print("File used:", csv_name)
print("Available columns:", df.columns.tolist())
display(df.head())

[PosixPath('/content/spacex_launches_cleaned.csv')]
Dataset loaded: (101, 11)
Available columns: ['name', 'status', 'success', 'date_utc', 'pad', 'rocket', 'details', 'links.article', 'links.webcast', 'links.wikipedia', 'year']


,name,status,success,date_utc,pad,rocket,details,links.article,links.webcast,links.wikipedia,year
0,Falcon Heavy | NROL-97,Launch Successful,True,2026-10-02 03:54:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,2026
1,Falcon 9 Block 5 | Transporter 18 (Dedicated S...,Launch Successful,True,2026-10-01 18:32:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,2026
2,Falcon 9 Block 5 | Crew-13,Launch Successful,True,2026-10-01 15:10:06+00:00,NaN,NaN,NaN,NaN,NaN,NaN,2026
3,Starship | Starlink Group 31-1 (Starship Fligh...,Launch Successful,True,2026-09-28 12:48:59+00:00,NaN,NaN,NaN,NaN,NaN,NaN,2026
4,Falcon 9 Block 5 | USSF-385,Launch Successful,True,2026-09-26 14:00:54+00:00,NaN,NaN,NaN,NaN,NaN,NaN,2026


KeyboardInterrupt: 

In [2]:
# --------------------------------------------------
# 2. Prepare launch-site information
# --------------------------------------------------

# Your CSV uses 'pad', not 'launchpad_name'.
# It contains a known site name but does not contain
# latitude/longitude or launchpad_id columns.

if "pad" not in df.columns:
    raise ValueError("The dataset does not contain a 'pad' column.")

# Coordinates for recognized SpaceX launch sites.
# Coordinates are used only when the site name matches.
known_sites = {
    "space launch complex 4e": {
        "name": "Space Launch Complex 4E",
        "latitude": 34.6321,
        "longitude": -120.6108,
        "locality": "Vandenberg Space Force Base, California"
    },
    "lc-39a": {
        "name": "Launch Complex 39A",
        "latitude": 28.6082,
        "longitude": -80.6043,
        "locality": "Kennedy Space Center, Florida"
    },
    "launch complex 39a": {
        "name": "Launch Complex 39A",
        "latitude": 28.6082,
        "longitude": -80.6043,
        "locality": "Kennedy Space Center, Florida"
    },
    "space launch complex 40": {
        "name": "Space Launch Complex 40",
        "latitude": 28.5619,
        "longitude": -80.5774,
        "locality": "Cape Canaveral, Florida"
    }
}

# Use actual pad values present in the dataset
pad_values = df["pad"].dropna().astype(str).unique()

site_records = {}

for pad in pad_values:
    normalized_pad = pad.strip().lower()

    if normalized_pad in known_sites:
        site_records[normalized_pad] = known_sites[normalized_pad]

print("\nRecognized launch sites:", len(site_records))
print("Non-missing pad records:", df["pad"].notna().sum())




Recognized launch sites: 1
Non-missing pad records: 1


In [4]:
# --------------------------------------------------
# 3. Create the interactive map
# --------------------------------------------------

import folium

if not site_records:
    print("\nNo recognized launch sites were found.")
    print("Actual pad values:", list(pad_values))
    print(
        "The uploaded dataset does not contain enough "
        "site information to place accurate markers."
    )

else:
    launch_map = folium.Map(
        location=[30.0, -100.0],
        zoom_start=4,
        tiles="OpenStreetMap"
    )

    site_summary = []

    for site_key, site in site_records.items():

        # Count only records explicitly associated with this pad.
        site_launches = df[
            df["pad"].astype("string").str.strip().str.lower()
            == site_key
        ]

        # Count known outcomes only
        if "success" in site_launches.columns:
            known_outcomes = pd.to_numeric(
                site_launches["success"], errors="coerce"
            ).dropna()

            success_rate = (
                f"{known_outcomes.mean() * 100:.1f}%"
                if len(known_outcomes) > 0
                else "N/A"
            )
        else:
            known_outcomes = pd.Series(dtype=float)
            success_rate = "N/A"

        site_name = site["name"]
        launch_count = len(site_launches)

        popup_html = f"""
        <div style="width:280px">
            <h4>{site_name}</h4>
            <b>Location:</b> {site["locality"]}<br>
            <b>Records explicitly linked to this site:</b>
            {launch_count}<br>
            <b>Known outcomes:</b> {len(known_outcomes)}<br>
            <b>Success rate among known outcomes:</b>
            {success_rate}
        </div>
        """

        folium.Marker(
            location=[site["latitude"], site["longitude"]],
            tooltip=site_name,
            popup=folium.Popup(popup_html, max_width=350),
            icon=folium.Icon(color="blue", icon="info-sign")
        ).add_to(launch_map)

        site_summary.append({
            "launchpad_name": site_name,
            "locality": site["locality"],
            "latitude": site["latitude"],
            "longitude": site["longitude"],
            "launch_records": launch_count,
            "known_outcomes": len(known_outcomes),
            "success_rate": success_rate
        })

    # Fit the map to the recognized markers
    coordinates = [
        [site["latitude"], site["longitude"]]
        for site in site_records.values()
    ]

    if len(coordinates) > 1:
        launch_map.fit_bounds(coordinates)

    print("\nLaunch-site summary:")
    display(pd.DataFrame(site_summary))

    print("\nInteractive launch-site map:")
    display(launch_map)



Launch-site summary:


,launchpad_name,locality,latitude,longitude,launch_records,known_outcomes,success_rate
0,Space Launch Complex 4E,"Vandenberg Space Force Base, California",34.6321,-120.6108,1,0,N/A



Interactive launch-site map:


In [6]:
# --------------------------------------------------
# 4. Save the map
# --------------------------------------------------

from pathlib import Path
from google.colab import files

output_path = Path("/content/spacex_launch_sites_map.html")
launch_map.save(str(output_path))

print("\nSaved interactive map:", output_path)
files.download(str(output_path))


Saved interactive map: /content/spacex_launch_sites_map.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Proximity analysis note
The markers locate launchpads. If your course requires distance/proximity analysis, calculate great-circle distances between coordinate pairs explicitly and explain the reference point. Do not infer proximity solely from the map display.